Simple chat bot 


Chat bot with summarized memory 
Create a LLM \
Node START \
Node - ChatNode (chats with the users )\
Node - Summizier (summizeries the messagses )\
Node - END\
Edge -  Summizer or END 


STATE(Message state and summary)

In [ ]:
#Import all the modules 
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState  , StateGraph , START , END
from langchain.messages import HumanMessage , SystemMessage  , RemoveMessage
from langgraph.checkpoint.memory import MemorySaver
import os , getpass
from pprint import pprint
from dotenv import load_dotenv


In [ ]:
def _set_env(var:str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

In [ ]:
_set_env("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = 'true'
os.environ["LANGCHAIN_PROJECT"] = "langgraph-testing"

Create a LLM


In [ ]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model =  "gemini-3-flash-preview"
)

Define a state 


In [ ]:
class State(MessagesState):
    summary :str

Chat functions 


In [ ]:
def call_model(state : State):
    "Returns the llm invoke method"
    #Get the summary is present 
    summary = state.get('summary' , "")
    if summary:
        #add a system prompt 
        summary_prompt =  f"This is the summary up to date : {summary}"
        messages = [SystemMessage(content=summary_prompt)] + state["messages"]
    else :
        messages = state['messages']
    #Invoke the llm model
    response = llm.invoke(messages)
    return {
        "messages" : [response]
    }




In [ ]:
def summarize_messages(state:State):
    "Summarizes the complete messages"

    summary = state.get("summary" , "")
    #Check weather the summary is present 
    if summary :
        #Add sumamry and the intruction 
        summary_messages = (
            f"This is the summary till date : {summary}"
            "Extend the summary by adding details from the lastest messages"
        ) 
    else :
        summary_messages =  "Create a summary of the lastest messages"
    

    messages = state['messages'] + [HumanMessage(content=summary_messages)]
    #Get the response 
    response = llm.invoke(messages)
    #Delete the messages 
    delete_messages = [RemoveMessage(id=m.id) for m in state['messages'][:-2]]

    return {
        "summary" : response.content , 
        "messages" : delete_messages
    }

In [ ]:
def summary_conditional_node(state : State):
    "Checks the conditon to summarize or to end the conversation"

    if len(state['messages']) > 6 :
        return "summarize_messages"
    return END

Create a Agent State 

In [ ]:
builder = StateGraph(State)
builder.add_node("summarize_messages",summarize_messages )
builder.add_node("chat_node" , call_model)


builder.add_edge(START , "chat_node")
builder.add_conditional_edges("chat_node" , summary_conditional_node , {"summarize_messages" :"summarize_messages" , 
                                                                        END : END})
builder.add_edge("summarize_messages" , END)

memory = MemorySaver()

agent = builder.compile(checkpointer=memory)

In [ ]:
agent

In [ ]:
config = {
    "configurable" : {
        "thread_id" : "1"
    }
}

In [ ]:
response = agent.invoke({
    "messages" :HumanMessage("I am Puvith")} , config)

In [ ]:
response

In [ ]:
response = agent.invoke({
    "messages" : [HumanMessage("What is my name ")] 
} , config)

In [ ]:
for i in response['messages'][:]:
    i.pretty_print()

In [ ]:
messages = [HumanMessage(content="Hi, my name is Puvith Kumar."),

HumanMessage(content="I am working at Endava as a software engineer."),

HumanMessage(content="I love building AI applications using LangGraph and LangChain."),

HumanMessage(content="I created a project called SareeFusion for saree design automation."),

HumanMessage(content="I also worked on a classroom monitoring system using depth cameras."),

HumanMessage(content="My favorite programming language is Python."),

HumanMessage(content="I enjoy backend development using Spring Boot."),

HumanMessage(content="I am interested in Generative AI and agentic workflows.")]

In [ ]:
agent.invoke({
    'messages' : messages
} , config)

In [ ]:
response = agent.invoke({ 
    "messages":
                         [HumanMessage(content="Where do I work?")] } , config) 

In [ ]:
response

In [ ]:
for i in response['messages'][:]:
    i.pretty_print()